## ARIA-PROMOTE - WP7 Validation


### Description:

Notebook developing evaluation and validation diagnostics for WP7 eORCA025_SPNA20 simulation.

### Contact:

Ollie Tooth (oliver.tooth@noc.ac.uk)

In [1]:
import cartopy.crs as ccrs
import icechunk
import matplotlib.pyplot as plt
import xarray as xr
from nemo_cookbook import NEMODataTree

### 1.1 Create combined domain_cfg + mesh_mask NetCDF files

In [ ]:
# === Domain and Mesh Mask Datasets === #
ds_domain = xr.open_dataset("/dssgfs01/working/otooth/PROMOTE/eORCA025_SPNA20/domain_cfg.nc")
ds_mesh_mask = xr.open_dataset("/dssgfs01/working/otooth/PROMOTE/eORCA025_SPNA20/mesh_mask.nc")
ds_domcfg = xr.merge([ds_domain, ds_mesh_mask], compat='override')

ds_1_domain = xr.open_dataset("/dssgfs01/working/otooth/PROMOTE/eORCA025_SPNA20/1_domain_cfg.nc")
ds_1_mesh_mask = xr.open_dataset("/dssgfs01/working/otooth/PROMOTE/eORCA025_SPNA20/1_mesh_mask.nc")
ds_1_domcfg = xr.merge([ds_1_domain, ds_1_mesh_mask], compat='override')

ds_domcfg

In [ ]:
# Define nests dictionary for NEMODataTree:
nests = {
    "1": {
    "parent": "/",
    "rx": 5,
    "ry": 5,
    "imin": 905,
    "imax": 1165,
    "jmin": 908,
    "jmax": 1155,
    "iperio": False
    }}

# Define datasets dictionary for NEMODataTree:
datasets = {
    "parent": {
        "domain": ds_domcfg,
        },
    "child": {
        "1":{"domain": ds_1_domcfg,
        }},
    }

nemo = NEMODataTree.from_datasets(
    datasets=datasets,
    nests=nests,
    iperio=True,
    nftype="T",
    read_mask=True,
    vco_ref=True,
    nbghost_child=None,
    keep_dom_vars=["bathy_metry"]
    )

nemo

In [ ]:
# Write NEMO global parent domain_cfg_mesh_mask to a NetCDF file:
ds_domcfg = nemo.to_domain_cfg(dom=".", vco_ref=False)
ds_domcfg["bathy_metry"] = ds_domain["bathy_metry"].squeeze(drop=True)
ds_domcfg.to_netcdf("/dssgfs01/working/otooth/PROMOTE/eORCA025_SPNA20/domain_cfg_mesh_mask.nc")

ds_domcfg

In [ ]:
# Write NEMO regional child domain_cfg_mesh_mask to a NetCDF file:
ds_1_domcfg = nemo.to_domain_cfg(dom="1", vco_ref=False)
ds_1_domcfg["bathy_metry"] = ds_1_domain["bathy_metry"].squeeze(drop=True)
ds_1_domcfg.to_netcdf("/dssgfs01/working/otooth/PROMOTE/eORCA025_SPNA20/1_domain_cfg_mesh_mask.nc")

ds_1_domcfg

### 1.2 Utility Function

* Define helper function to open Virtual NEMODataTrees from local Icechunk repositories.

* **To do:** Add notes on tips & tricks for virtualizing (dropping variables and loadable variables).

* **To do:** Add notes on creating combined domain_cfg + mesh_masks for virtualizing nested NEMODataTrees.

In [2]:
def open_virtual_NEMODataTree(
        repo_path: str,
        name: str,
        branch: str = "main",
        iperio: bool = True,
        nftype: str = "T",
    ) -> NEMODataTree:
    # Open Icechunk repository to identify virtual chunk container prefix path:
    repo_ini = icechunk.Repository.open(storage=icechunk.local_filesystem_storage(repo_path))
    store_path = next(iter(repo_ini.config.virtual_chunk_containers.keys()))

    # Authorise Icechunk to fetch virtual chunks from the original NetCDF files on our local filesystem:
    credentials = icechunk.credentials.containers_credentials({store_path: icechunk.credentials.LocalFileSystemAccess})

    # Open Icechunk repository stored & authorise virtual chunk access:
    repo = icechunk.Repository.open(storage=icechunk.local_filesystem_storage(repo_path),
                                                authorize_virtual_chunk_access=credentials
                                                )

    # Create virtual NEMODataTree from Icechunk repository:
    nemo = NEMODataTree.from_icechunk(repo=repo,
                                               branch=branch,
                                               iperio=iperio,
                                               nftype=nftype,
                                               name=name,
                                               )

    return nemo

### Open PROMOTE eORCA025 SPNA20 nested NEMODataTree:

In [3]:
nemo = open_virtual_NEMODataTree(
    repo_path="/dssgfs01/working/otooth/PROMOTE/repos/eORCA025_SPNA20_1m",
    name="eORCA025 SPNA20 JRA55-do",
    iperio=True,
    nftype="T",
)

nemo["gridT"].dataset = nemo["gridT"].dataset.rename({"thkcello": "e3t"})
nemo["gridT/1_gridT"].dataset = nemo["gridT/1_gridT"].dataset.rename({"thkcello": "e3t"})

nemo

  2026-11-04T13:20:11.093380Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:343

  2026-11-04T13:20:11.192102Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:343



<xarray.DataTree 'eORCA025 SPNA20 JRA55-do'>
Group: /
│   Attributes:
│       nftype:   T
│       iperio:   True
├── Group: /gridU
│   │   Dimensions:               (j: 1206, i: 1440, time_counter: 624, k: 75,
│   │                              axis_nbounds: 2)
│   │   Coordinates:
│   │     * j                     (j) int64 10kB 1 2 3 4 5 ... 1202 1203 1204 1205 1206
│   │     * i                     (i) float64 12kB 1.5 2.5 3.5 ... 1.44e+03 1.44e+03
│   │       gphiu                 (j, i) float64 14MB dask.array<chunksize=(1206, 1440), meta=np.ndarray>
│   │       glamu                 (j, i) float64 14MB dask.array<chunksize=(1206, 1440), meta=np.ndarray>
│   │     * time_counter          (time_counter) datetime64[ns] 5kB 1958-01-16T12:00:...
│   │       time_centered         (time_counter) datetime64[ns] 5kB dask.array<chunksize=(1,), meta=np.ndarray>
│   │     * k                     (k) int64 600B 1 2 3 4 5 6 7 ... 69 70 71 72 73 74 75
│   │       depthu                (k) float32 300B dask.array<chunksize=(75,), meta=np.ndarray>
│   │   Dimensions without coordinates: axis_nbounds
│   │   Data variables:
│   │       e2u                   (j, i) float64 14MB dask.array<chunksize=(1206, 1440), meta=np.ndarray>
│   │       e3u                   (time_counter, k, j, i) float32 325GB dask.array<chunksize=(1, 3, 1206, 1440), meta=np.ndarray>
│   │       depthu_bounds         (time_counter, k, axis_nbounds) float32 374kB dask.array<chunksize=(1, 75, 2), meta=np.ndarray>
│   │       e1u                   (j, i) float64 14MB dask.array<chunksize=(1206, 1440), meta=np.ndarray>
│   │       tauuo                 (time_counter, j, i) float32 4GB dask.array<chunksize=(1, 1206, 1440), meta=np.ndarray>
│   │       sozocrtx              (time_counter, j, i) float32 4GB dask.array<chunksize=(1, 1206, 1440), meta=np.ndarray>
│   │       umask                 (k, j, i) int8 130MB dask.array<chunksize=(75, 1206, 1440), meta=np.ndarray>
│   │       uo                    (time_counter, k, j, i) float32 325GB dask.array<chunksize=(1, 3, 1206, 1440), meta=np.ndarray>
│   │       time_counter_bounds   (time_counter, axis_nbounds) datetime64[ns] 10kB dask.array<chunksize=(1, 2), meta=np.ndarray>
│   │       umaskutil             (j, i) int8 2MB dask.array<chunksize=(1206, 1440), meta=np.ndarray>
│   │       time_centered_bounds  (time_counter, axis_nbounds) datetime64[ns] 10kB dask.array<chunksize=(1, 2), meta=np.ndarray>
│   │   Attributes:
│   │       name:         OUTPUT/eORCA025_SPNA20_1m_grid_U
│   │       description:  monthly vector ocean variables defined on NEMO curvilinear ...
│   │       title:        monthly vector ocean variables defined on NEMO curvilinear ...
│   │       Conventions:  CF-1.6
│   └── Group: /gridU/1_gridU
│           Dimensions:               (j1: 1243, i1: 1308, time_counter: 624, k1: 75,
│                                      axis_nbounds: 2, k: 75)
│           Coordinates:
│             * j1                    (j1) int64 10kB 1 2 3 4 5 ... 1239 1240 1241 1242 1243
│               j_j1                  (j1) float64 10kB dask.array<chunksize=(1243,), meta=np.ndarray>
│             * i1                    (i1) float64 10kB 1.5 2.5 3.5 ... 1.308e+03 1.308e+03
│               1_glamu               (j1, i1) float64 13MB dask.array<chunksize=(1243, 1308), meta=np.ndarray>
│               1_gphiu               (j1, i1) float64 13MB dask.array<chunksize=(1243, 1308), meta=np.ndarray>
│               i_i1                  (i1) float64 10kB dask.array<chunksize=(1308,), meta=np.ndarray>
│               time_centered         (time_counter) datetime64[ns] 5kB dask.array<chunksize=(1,), meta=np.ndarray>
│             * k1                    (k1) int64 600B 1 2 3 4 5 6 7 ... 69 70 71 72 73 74 75
│               1_depthu              (k1) float32 300B dask.array<chunksize=(75,), meta=np.ndarray>
│               depthu                (k) float32 300B dask.array<chunksize=(75,), meta=np.ndarray>
│           Dimensions without coo

In [ ]:
# Define subpolar North Atlantic boundary box for analysis:
bbox_spna = (-65, 0, 42, 88)  # (-65°E, -0°E, 42°N, 88°N)

speed_parent = (nemo["gridU/uo"].interp_to(to="T")**2 + nemo["gridV/vo"].interp_to(to="T")**2 )**0.5
speed_child = (nemo["gridU/1_gridU/uo"].interp_to(to="T")**2 + nemo["gridV/1_gridV/vo"].interp_to(to="T")**2 )**0.5

plt.figure()
# Plot the surface current speed (m/s) in the subpolar North Atlantic region:
(speed_child
 .isel(time_counter=120, k1=0)
 .geoplot(projection=ccrs.Orthographic(central_longitude=-32, central_latitude=55),
          extent=bbox_spna,
          cmap='Blues_r',
          clabel_kwargs={'label': 'Surface Current Speed (m/s)'},
          vmin=0,
          vmax=1.2,
          cbar_kwargs={'extend': "max"}
          )
 )
plt.title('Surface Current Speed\nTime-Mean [eORCA025 + SPNA20]', fontsize=13, fontweight='bold')

In [ ]:
nemo